# Model B Multi-Input DRAFT

## 1. Setup

In [ ]:
import os, json, ast, re, random, shutil
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from tqdm.auto import tqdm
import torch

DATA_ROOT = Path(os.environ.get('VINDR_ROOT', 'data/vindr_mammo'))
RAW       = DATA_ROOT / 'vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0'
PNG       = DATA_ROOT / 'png'
ANN       = DATA_ROOT / 'annotations_processed'
LORA_DATA = DATA_ROOT / 'lora_data'
LORA_RUNS = DATA_ROOT / 'lora_runs'
LORA_DATA_MULTI = LORA_DATA / 'multiimage_data' 
LORA_RUNS_MULTI = LORA_RUNS / 'multiimage_runs'
YOLO_DIR  = DATA_ROOT / 'yolo'
RUNS      = DATA_ROOT / 'runs'

for d in [ANN, LORA_DATA, LORA_DATA_MULTI, LORA_RUNS, LORA_RUNS_MULTI, YOLO_DIR, RUNS]:
    d.mkdir(parents=True, exist_ok=True)

BREAST_CSV  = RAW / 'breast-level_annotations.csv'
FINDING_CSV = RAW / 'finding_annotations.csv'

breast_df  = pd.read_csv(BREAST_CSV)
finding_df = pd.read_csv(FINDING_CSV)

# drop one pair of corrupt images (R/L MLOs) for now; may drop whole pt if extended to standard 4 mammography inputs
BAD_IDS = {'3e71b089f6e55d5791771b05f88518dd', 'd1933b9b47e13ddc6d9c92febcf3c1bd'} 
breast_df  = breast_df[~breast_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)
finding_df = finding_df[~finding_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)

print('Breast rows:', len(breast_df), ' Finding rows:', len(finding_df))
print('PNG files:', len(list(PNG.glob('*.png'))))
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

Breast rows: 19998  Finding rows: 20484
PNG files: 19999
CUDA: True | NVIDIA H200 NVL


## 2. Grouped labels

| Group | Includes | Why |
|---|---|---|
| Mass | Mass | 1,123+ |
| Suspicious Calcification | Suspicious Calcification | 402+ |
| Asymmetry | Asymmetry, Focal Asymmetry, Global Asymmetry | clinically related, individually small |
| Architectural Distortion | Architectural Distortion | important cancer sign |
| Suspicious Lymph Node | Suspicious Lymph Node | clinically meaningful |
| Other Suspicious Signs | Skin Thickening, Skin Retraction, Nipple Retraction | too rare individually |

In [ ]:
GROUPS = {
    'Mass':                     ['Mass'],
    'Suspicious Calcification': ['Suspicious Calcification'],
    'Asymmetry':                ['Asymmetry', 'Focal Asymmetry', 'Global Asymmetry'],
    'Architectural Distortion': ['Architectural Distortion'],
    'Suspicious Lymph Node':    ['Suspicious Lymph Node'],
    'Other Suspicious Signs':   ['Skin Thickening', 'Skin Retraction', 'Nipple Retraction'],
}

GROUPED_CLASSES = list(GROUPS.keys())

GROUP_TO_PHRASE = {
    'Mass': 'A suspicious mass is present',
    'Suspicious Calcification': 'Suspicious calcifications are present',
    'Asymmetry': 'An asymmetry is present',
    'Architectural Distortion': 'Architectural distortion is present',
    'Suspicious Lymph Node': 'A suspicious lymph node is present',
    'Other Suspicious Signs': 'Other suspicious mammographic signs are present',
}

GROUP_TO_KNOWLEDGE = {
    'Mass': 'A mammographic mass is a space-occupying lesion with a visible shape, margin, and density.',
    'Suspicious Calcification': 'Suspicious calcifications appear as small high-density bright foci, often grouped or clustered.',
    'Asymmetry': 'Asymmetry is an area of denser breast tissue without a clear mass, possibly focal or global.',
    'Architectural Distortion': 'Architectural distortion appears as pulling or disruption of normal breast tissue without a definite mass.',
    'Suspicious Lymph Node': 'A suspicious lymph node appears enlarged, dense, rounded, or with abnormal morphology.',
    'Other Suspicious Signs': 'Other suspicious signs include skin thickening, nipple retraction, or skin retraction.',
}

# fix inconsistencies in the finding_categories col of finding_df
def parse_finding_categories(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        v = ast.literal_eval(str(x))
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [str(x)]

# takes clean finding list and maps it to the classes defined in GROUPS, returning a sorted list of unique groups
def map_to_groups(categories):
    cats = parse_finding_categories(categories)
    if 'No Finding' in cats:
        return []
    out = set()
    for g, raws in GROUPS.items():
        for r in raws:
            if r in cats:
                out.add(g)
    return sorted(out)

# maps each image_id to a set of corresponding group labels using map_to_groups. 
# tells WHAT is in the image -- used for global image classification/filtering.
image_to_groups = defaultdict(set)

# loops through each row, passes row's categories to map_to_groups, and adds the result to the image's set
for _, r in finding_df.iterrows():
    for g in map_to_groups(r['finding_categories']):
        image_to_groups[r['image_id']].add(g)
# converts sets to sorted lists. result is a dict structured like {'image_01': ['Mass', 'Asymmetry'], ...}
image_to_groups = {k: sorted(v) for k, v in image_to_groups.items()}

"""
parse_finding_categories / map_to_groups / image_to_groups: 
build a quick-reference dictionary where you can look up any image_id 
and get a sorted, unique list of the high-level categories present in that image
"""

# counts the number of images in each group and the number of normal images (no findings)
counts = Counter()
for groups in image_to_groups.values():
    for g in groups:
        counts[g] += 1

n_normal = len(breast_df) - sum(1 for img in breast_df['image_id'] if img in image_to_groups and len(image_to_groups[img]) > 0)
print(f'{"Normal (no finding)":30s} {n_normal}')
for g in GROUPED_CLASSES:
    print(f'{g:30s} {counts[g]}')

Normal (no finding)            18230
Mass                           1113
Suspicious Calcification       442
Asymmetry                      390
Architectural Distortion       119
Suspicious Lymph Node          53
Other Suspicious Signs         86


## 3. Splits

In [ ]:
SEED = 0
VAL_FRAC = 0.10

train_studies_all = sorted(breast_df.loc[breast_df['split'] == 'training', 'study_id'].unique().tolist())
test_studies      = sorted(breast_df.loc[breast_df['split'] == 'test',     'study_id'].unique().tolist())

# carves out a validation set from the training studies
rng = random.Random(SEED)
rng.shuffle(train_studies_all)
n_val = int(len(train_studies_all) * VAL_FRAC)
val_studies   = sorted(train_studies_all[:n_val])
train_studies = sorted(train_studies_all[n_val:])

# returns all image_ids corresponding to the given study_ids
def studies_to_images(study_ids):
    return sorted(breast_df.loc[breast_df['study_id'].isin(study_ids), 'image_id'].unique().tolist())

train_ids = studies_to_images(train_studies)
val_ids   = studies_to_images(val_studies)
test_ids  = studies_to_images(test_studies)

print(f'Train studies: {len(train_studies):5d}  images: {len(train_ids):5d}')
print(f'Val   studies: {len(val_studies):5d}  images: {len(val_ids):5d}')
print(f'Test  studies: {len(test_studies):5d}  images: {len(test_ids):5d}')

assert not (set(train_studies) & set(val_studies))
assert not (set(train_studies) & set(test_studies))
assert not (set(val_studies)   & set(test_studies))

(ANN / 'splits.json').write_text(json.dumps({
    'train_studies': train_studies, 'val_studies': val_studies, 'test_studies': test_studies,
    'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids,
}))
print('Saved splits to', ANN / 'splits.json')

Train studies:  3600  images: 14398
Val   studies:   400  images:  1600
Test  studies:  1000  images:  4000
Saved splits to $VINDR_ROOT/annotations_processed/splits.json


## 4. Box Scaling

In [ ]:
_png_dim_cache = {}

def get_png_size(image_id):
    if image_id in _png_dim_cache:
        return _png_dim_cache[image_id]
    p = PNG / f'{image_id}.png'
    if not p.exists():
        return None
    with PILImage.open(p) as im:
        _png_dim_cache[image_id] = im.size
    return _png_dim_cache[image_id]

def box_to_norm1000(row, image_id):
    """Original DICOM coords -> PNG-scaled -> normalized 0-1000.
    Returns [x1, y1, x2, y2] ints, or None if invalid."""
    size = get_png_size(image_id)
    if size is None: return None
    png_w, png_h = size

    orig_w, orig_h = float(row['width']), float(row['height'])
    sx, sy = png_w / orig_w, png_h / orig_h

    x1 = float(row['xmin']) * sx
    y1 = float(row['ymin']) * sy
    x2 = float(row['xmax']) * sx
    y2 = float(row['ymax']) * sy

    x1 = max(0, min(png_w, x1)); x2 = max(0, min(png_w, x2))
    y1 = max(0, min(png_h, y1)); y2 = max(0, min(png_h, y2))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None

    return [
        int(round(x1 / png_w * 1000)),
        int(round(y1 / png_h * 1000)),
        int(round(x2 / png_w * 1000)),
        int(round(y2 / png_h * 1000)),
    ]

'''
Build image_id -> list of {group, box} with PNG-scaled normalized coords.
- WHAT type of finding it is (group) 
- WHERE each finding is (box)
- WHICH view it's on
'''

image_to_boxes = defaultdict(list)
_side_lookup = breast_df.set_index('image_id')['laterality']
_view_lookup  = breast_df.set_index('image_id')['view_position']

for _, r in tqdm(finding_df.iterrows(), total=len(finding_df), desc='Scaling boxes'):
    if pd.isna(r.get('xmin', np.nan)): continue
    # maps finding cats to groups and normalizes the bounding box coords
    groups = map_to_groups(r['finding_categories'])
    if not groups: continue
    box = box_to_norm1000(r, r['image_id'])
    if box is None: continue
    if r['image_id'] not in _view_lookup.index: continue
    laterality = _side_lookup.loc[r['image_id']]
    view_position = _view_lookup.loc[r['image_id']]
    # appends group, box, and view_position to corresponding image_id
    for g in groups:
        image_to_boxes[r['image_id']].append(
            {'group': g, 'box': box, 'laterality': laterality, 'view_position': view_position}
        )

print(f'Images with at least one grouped box: {len(image_to_boxes)}')

sample_id = next(iter(image_to_boxes))
print(f'\nSample {sample_id}: {image_to_boxes[sample_id]}')

Images with at least one grouped box: 1768

Sample <image_id>: [{'group': 'Mass', 'box': [841, 492, 887, 527], 'laterality': 'R', 'view_position': 'CC'}]


## 5. Build Groups

each view pair group = 
- its 2 MammoView's (CC or MLO of both sides)
- every finding across both views, each tagged with which view it came from

In [ ]:
from dataclasses import dataclass, field

_LATERALITY_WORD = {'L': 'left', 'R': 'right'}
_SIDE_ORDER = ['L', 'R']  # consistent order

# each instance is an individual image in the dataset
@dataclass(frozen=True)
class MammoView:
    image_id: str
    study_id: str
    laterality: str          # 'L' or 'R'
    view_position: str       # 'CC' or 'MLO'
    png_path: Path
    width: int
    height: int
    birads: str
    density: str
    split: str

# each instance is an annotated finding with its classification/location/ view association
@dataclass(frozen=True)
class FindingBox:
    group: str          # finding classification
    box: tuple           # (x1, y1, x2, y2), normalized 0-1000
    laterality: str      # 'L' or 'R' (which side this box is from)

# combines MammoView and FindingBox (one object = CC and MLO images of one side of the pt's breast with all their corresponding findings)
@dataclass
class ViewPairGroup:
    '''
    one object = L and R image of the SAME view (CC or MLO) + every finding across both, each tagged with which side it's on
    '''
    study_id: str
    view_position: str
    views: list = field(default_factory=list) # list of MammoView objects
    findings: list = field(default_factory=list) # list of finding_box objects

    @property
    def group_id(self):
        return f'{self.study_id}_{self.view_position}'

    @property
    def is_complete_pair(self):
        sides = {v.laterality for v in self.views}
        return {'L', 'R'}.issubset(sides)

    @property
    def split(self):
        return self.views[0].split

    def view_by_laterality(self, laterality):
        for v in self.views:
            if v.laterality == laterality:
                return v
        return None

    # unique finding types present on this breast side
    def unique_groups(self):
        return sorted({f.group for f in self.findings})


def build_view_pair_groups(breast_df, image_to_boxes):
    """
    Reshapes per-image data (breast_df rows, image_to_boxes entries) into
    per-(study_id, view_position) L/R pairs.
    """
    view_records = {}
    for _, row in breast_df.iterrows():
        view_records[row['image_id']] = MammoView(
            image_id=row['image_id'], study_id=row['study_id'],
            laterality=row['laterality'], view_position=row['view_position'],
            png_path=PNG / f"{row['image_id']}.png",
            width=int(row['width']), height=int(row['height']),
            birads=str(row['breast_birads']), density=str(row['breast_density']),
            split=row['split'],
        )

    groups = {}
    for v in view_records.values():
        key = (v.study_id, v.view_position)
        if key not in groups:
            groups[key] = ViewPairGroup(study_id=v.study_id, view_position=v.view_position)
        groups[key].views.append(v)

    for g in groups.values():
        g.views.sort(key=lambda v: _SIDE_ORDER.index(v.laterality) if v.laterality in _SIDE_ORDER else 99)

    n_total = len(groups)
    groups = {k: g for k, g in groups.items() if g.is_complete_pair}
    n_dropped_groups = n_total - len(groups)

    # attach bounding box objects to group
    for image_id, entries in image_to_boxes.items():
        view = view_records.get(image_id)
        if view is None: continue
        key = (view.study_id, view.view_position)
        if key not in groups: continue
        for e in entries:
            groups[key].findings.append(FindingBox(group=e['group'], box=e['box'], laterality=view.laterality))

    # count complete sets
    result = list(groups.values())
    print(f'Built {len(result)} complete L+R view-pair groups '
          f'(dropped {n_dropped_groups} groups with no match)')
    return result

view_pair_groups = build_view_pair_groups(breast_df, image_to_boxes)

# keep all groups from the same patient in the same split
by_study = defaultdict(list)
for g in view_pair_groups:
    by_study[g.study_id].append(g)

train_groups = [g for sid in train_studies for g in by_study[sid]]
val_groups   = [g for sid in val_studies   for g in by_study[sid]]
test_groups  = [g for sid in test_studies  for g in by_study[sid]]

print(f'Train groups: {len(train_groups)}  Val groups: {len(val_groups)}  Test groups: {len(test_groups)}')

Built 9998 complete L+R view-pair groups (dropped 1 groups with no match
Train groups: 7198  Val groups: 800  Test groups: 2000


## 6. Build Message
- view order is always [L, R]
- each box in target text tagged with which image it belongs to
- each user turn explicitly states the views are of the same patient
- asymmetry is reported per side

In [6]:
SYSTEM_MAMMO_MULTIIMAGE = (
    'You are an expert radiology assistant for mammography. '
    'You will be shown the same mammographic view (either CC or MLO) of the LEFT and RIGHT breast of the same patient.'
    'Answer carefully using only the image information provided.'
)

def _ordered_views(group):
    present = {v.laterality: v for v in group.views}
    return [present[side] for side in _SIDE_ORDER if side in present]

def patient_context_sentence(group):
    return (f'Image 1 and Image 2 are the left and right {group.view_position} views, '
            f'respectively, of the same patient.')

def make_multiimage_messages(group, user_text, assistant_text):
    ordered = _ordered_views(group)
    user_content = [{'type': 'image', 'image': str(v.png_path)} for v in ordered]
    user_content.append({'type': 'text', 'text': f'{patient_context_sentence(group)} {user_text}'})
    return [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_MAMMO_MULTIIMAGE}]},
        {'role': 'user',      'content': user_content},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
    ]
 
def _image_index_label(group, laterality):
    ordered = _ordered_views(group)
    v = group.view_by_laterality(laterality)
    if v is None:
        raise ValueError(f'laterality {laterality!r} not found in group {group.group_id}')
    return f'Image {ordered.index(v) + 1} ({_LATERALITY_WORD[laterality]})'

# checks if groups is empty. if not, lists the suspicious findings present
# only cares about the localized physical findings
def finding_classification_answer(group):
    other_groups = sorted(g for g in group.unique_groups() if g != 'Asymmetry')
    asym_sides = sorted({f.laterality for f in group.findings if f.group == 'Asymmetry'})

    parts = []
    if other_groups:
        parts.append('The suspicious mammographic findings present are: ' + ', '.join(other_groups) + '.')
    if asym_sides:
        sides = ' and '.join(_LATERALITY_WORD[s] for s in asym_sides)
        parts.append(f'An asymmetry is present in the {sides} breast.')
    if not parts:
        return 'No suspicious mammographic finding is present.'
    return ' '.join(parts)

# BI-RADS/density are assigned per breast. L/R pairing -> report both sides explicitly
def birads_density_answer(group):
    parts = []
    for v in _ordered_views(group):
        side = _LATERALITY_WORD[v.laterality]
        parts.append(f'The {side} breast is BI-RADS category {v.birads} with density {v.density}.')
    return ' '.join(parts)

def grounded_report_answer(group):
    parts = [birads_density_answer(group)]
    if not group.findings:
        parts.append('No suspicious mammographic finding is present.')
        return ' '.join(parts)
    for f in group.findings:
        label = _image_index_label(group, f.laterality)
        b = f.box
        if f.group == 'Asymmetry':
            side = _LATERALITY_WORD[f.laterality]
            parts.append(f'An asymmetry is present in the {side} breast, {label}, '
                          f'at <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
        else:
            parts.append(f'{GROUP_TO_PHRASE[f.group]} in {label} <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
    return ' '.join(parts)

def single_finding_grounding_answer(group, finding):
    label = _image_index_label(group, finding.laterality)
    b = finding.box
    if finding.group == 'Asymmetry':
        side = _LATERALITY_WORD[finding.laterality]
        user = f'Compare the two images and locate any asymmetry. {GROUP_TO_KNOWLEDGE["Asymmetry"]}'
        assistant = f'An asymmetry is present in the {side} breast, {label}, at <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.'
    else:
        user = f'Locate the {finding.group} finding. {GROUP_TO_KNOWLEDGE[finding.group]}'
        assistant = f'The {finding.group} is located in {label} at <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.'
    return user, assistant

## test

In [ ]:
# visually inspect 5 training view-pairs to confirm boxes align with the correct image AND side (L vs R)
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image as PILImage

def denorm_box(box, W, H):
    """box is [x1, y1, x2, y2] normalized 0-1000 -> pixel coords for this image."""
    x1, y1, x2, y2 = box
    return (x1 / 1000 * W, y1 / 1000 * H, x2 / 1000 * W, y2 / 1000 * H)

GROUP_COLORS = {
    'Mass':                     'red',
    'Suspicious Calcification': 'yellow',
    'Asymmetry':                'cyan',
    'Architectural Distortion': 'magenta',
    'Suspicious Lymph Node':    'lime',
    'Other Suspicious Signs':   'orange',
}

# prioritize groups that actually have findings
groups_with_findings = [g for g in train_groups if g.findings]
sample_groups = groups_with_findings[:5]

if len(sample_groups) < 5:
    print(f'Only {len(sample_groups)} train groups have findings -- showing all of them.')

for group in sample_groups:
    ordered = _ordered_views(group)  # [left_view, right_view]
    fig, axes = plt.subplots(1, 2, figsize=(12, 8))

    for ax, view in zip(axes, ordered):
        img = PILImage.open(view.png_path)
        W, H = img.size
        ax.imshow(img, cmap='gray')
        ax.set_title(f'{_LATERALITY_WORD[view.laterality].capitalize()} ({view.view_position})\n{view.image_id}',
                     fontsize=10, pad=12)

        # only draw boxes that belong to THIS side
        for f in group.findings:
            if f.laterality != view.laterality:
                continue
            x1, y1, x2, y2 = denorm_box(f.box, W, H)
            color = GROUP_COLORS.get(f.group, 'white')
            ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2,
                                    edgecolor=color, facecolor='none'))
            ax.text(x1, y1 - 5, f.group, color=color, fontsize=9,
                    bbox=dict(facecolor='black', alpha=0.7, pad=1))
        ax.axis('off')

    fig.suptitle(f'{group.group_id}  |  findings: {group.unique_groups()}', fontsize=12, y=0.98)    
    plt.tight_layout(rect=[0, 0, 1, 0.90])
    plt.show()

## 7. Build JSONLs
same 4-task structure as Model B:
1. `finding_classification` — 1 example (findings across all views)
2. `birads_density` — 1 example
3. `grounded_report` — 1 example (all findings, each box tagged with source image)
4. `single_finding_grounding` — 1 example PER finding

In [27]:
# formats images into a list of multiple different training examples
def build_examples_for_group(group):
    if not group.views:
        return []
    base_meta = {
        'group_id': group.group_id, 'study_id': group.study_id, 'view_position': group.view_position,
        'is_complete_pair': group.is_complete_pair,
        'images': [str(v.png_path) for v in _ordered_views(group)],
    }
    examples = []

    # Task 1: finding classification
    examples.append({**base_meta, 'task': 'finding_classification',
        'messages': make_multiimage_messages(group,
            'What suspicious mammographic findings are present, and in which breast?',
            finding_classification_answer(group))})

    # Task 2: BI-RADS + density (per side)
    examples.append({**base_meta, 'task': 'birads_density',
        'messages': make_multiimage_messages(group,
            'What is the BI-RADS category and breast density for each side?',
            birads_density_answer(group))})

    # Task 3: grounded report
    examples.append({**base_meta, 'task': 'grounded_report',
        'messages': make_multiimage_messages(group,
            'Generate a grounded mammography report comparing the left and right images. '
            'For each suspicious finding, state which image and side it appears in and append its '
            'location as <box>x_min y_min x_max y_max</box> in normalized 0-1000 coordinates.',
            grounded_report_answer(group))})

    # Task 4: per-finding grounding
    for finding in group.findings:
        user_instr, assistant_text = single_finding_grounding_answer(group, finding)
        examples.append({**base_meta, 'task': 'single_finding_grounding',
            'messages': make_multiimage_messages(group, user_instr, assistant_text)})

    return examples

# for the specified split (train/val/test), iterates through the ViewPairGroups, builds examples for each, and writes them to a jsonl file. 
def write_split(groups, split_name, out_dir):
    out_path = out_dir / f'{split_name}.jsonl'
    n_groups = n_ex = 0
    with open(out_path, 'w') as f:
        for group in tqdm(sorted(groups, key=lambda g: g.group_id), desc=f'{out_dir.name}/{split_name}'):
            exs = build_examples_for_group(group)
            if not exs:
                continue
            n_groups += 1
            for ex in exs:
                f.write(json.dumps(ex) + '\n')
                n_ex += 1
    print(f'{split_name}: groups={n_groups}, examples={n_ex}')

MULTI_GND_DIR = LORA_DATA_MULTI / 'grounded_multiimage'
MULTI_GND_DIR.mkdir(parents=True, exist_ok=True)

print('=== Multi-image grounded (Model B extension) ===')
for split_name, groups in [('train', train_groups), ('val', val_groups), ('test', test_groups)]:
    write_split(groups, split_name, MULTI_GND_DIR)

=== Multi-image grounded (Model B extension) ===


train: groups=7198, examples=23319


val: groups=800, examples=2622


test: groups=2000, examples=6482


## 8: Train Qwen2.5-VL-7B — Model B (multi-image)

In [ ]:
import torch, json, gc
from pathlib import Path
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'

def load_jsonl(path):
    return Dataset.from_list([json.loads(l) for l in open(path)])

DATA_DIR = MULTI_GND_DIR
OUT_DIR  = LORA_RUNS_MULTI / 'qwen25vl7b_grounded_multiimage'
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_ds = load_jsonl(DATA_DIR / 'train.jsonl')
val_ds   = load_jsonl(DATA_DIR / 'val.jsonl')

print('Model:', MODEL_ID, '| Data:', DATA_DIR)
print('Train:', len(train_ds), 'Val:', len(val_ds))
print('Sample assistant text:', train_ds[0]['messages'][-1]['content'][0]['text'][:200])

In [ ]:
# load processor + model (bf16, no quant -- H200 has the VRAM)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.config.use_cache = False
print('Loaded.', 'Params:', sum(p.numel() for p in model.parameters()) / 1e9, 'B')

In [ ]:
def clean_messages(messages):
    """Remove None fields that HF Datasets adds during schema unification."""
    out = []
    for m in messages:
        new_content = []
        for c in m['content']:
            cleaned = {k: v for k, v in c.items() if v is not None}
            new_content.append(cleaned)
        out.append({'role': m['role'], 'content': new_content})
    return out

def collate_qwen(examples):
    messages = [clean_messages(ex['messages']) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
    image_inputs, video_inputs = process_vision_info(messages)
    batch = processor(text=texts, images=image_inputs, videos=video_inputs,
                      padding=True, return_tensors='pt')
    labels = batch['input_ids'].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    if image_token_id is not None:
        labels[labels == image_token_id] = -100
    batch['labels'] = labels
    return batch

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

cfg = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to='none',
    dataset_kwargs={'skip_prepare_dataset': True},
)

trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=train_ds, eval_dataset=val_ds,
    peft_config=lora,
    data_collator=collate_qwen,
    processing_class=processor,
)
trainer.train()
trainer.save_model(str(OUT_DIR))
print('Saved to', OUT_DIR)

## 10. Inference

In [ ]:
from peft import PeftModel

ADAPTER_DIR = LORA_RUNS_MULTI / 'qwen25vl7b_grounded_multiimage'
EVAL_DATA   = MULTI_GND_DIR / 'test.jsonl'
PRED_OUT    = LORA_RUNS_MULTI / 'preds_grounded_multiimage.jsonl'

processor = AutoProcessor.from_pretrained(MODEL_ID)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('Loaded adapter:', ADAPTER_DIR)

In [ ]:
@torch.no_grad()
def generate_one(messages, max_new_tokens=256):
    inf_messages = [m for m in messages if m['role'] != 'assistant']
    text = processor.apply_chat_template(inf_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(inf_messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = gen[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

test_examples = [json.loads(l) for l in open(EVAL_DATA)]
print(f'{len(test_examples)} test examples')

with open(PRED_OUT, 'w') as f:
    for ex in tqdm(test_examples, desc='Generating'):
        pred = generate_one(ex['messages'])
        gt = ex['messages'][-1]['content'][0]['text']
        out = {
            'images': ex['images'],   # FIX: multi-image examples store a list, not ex['image']
            'task': ex['task'],
            'gt': gt,
            'pred': pred,
        }
        f.write(json.dumps(out) + '\n')

print('Saved predictions to', PRED_OUT)

## 11. Evaluation

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, cohen_kappa_score

preds = [json.loads(l) for l in open(PRED_OUT)]
by_task = defaultdict(list)
for p in preds: by_task[p['task']].append(p)
for t, items in by_task.items():
    print(f'{t:30s} {len(items)}')

GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

def extract_findings(text):
    t = text.lower()
    out = set()
    for g, kws in GROUP_KEYWORDS.items():
        if any(k in t for k in kws):
            out.add(g)
    return out

def to_binary_vec(groups):
    return [1 if g in groups else 0 for g in GROUPED_CLASSES]

# \b added: bilateral targets say "...with density DENSITY B." and without the word boundary
# the old pattern grabbed the D in "DENSITY" -> every case parsed as density D (see tests/test_metrics.py)
BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])\b', re.I)
DENSITY_RE = re.compile(r'density\s*([abcd])\b', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')

def extract_birads(text):
    m = BIRADS_RE.search(text)
    return m.group(1) if m else None

def extract_density(text):
    m = DENSITY_RE.search(text)
    return m.group(1).upper() if m else None

def extract_boxes(text):
    return [list(map(int, m)) for m in BOX_RE.findall(text)]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

print('Eval helpers loaded.')

In [ ]:
# finding classification metrics
if 'finding_classification' in by_task:
    items = by_task['finding_classification']
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'\nFinding classification: macro F1 = {macro_f1:.3f}, micro F1 = {micro_f1:.3f}')
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(len(GROUPED_CLASSES))))
    for i, g in enumerate(GROUPED_CLASSES):
        print(f'  {g:30s} P={p[i]:.3f}  R={r[i]:.3f}  F1={f[i]:.3f}')

# BI-RADS + density
if 'birads_density' in by_task:
    items = by_task['birads_density']
    bg, bp, dg, dp = [], [], [], []
    # two breasts per example now -> score each side separately
    # (the single-image version only ever read the first BI-RADS in the text = left side only)
    def _side_sentence(text, word):
        for s in re.split(r'(?<=\.)\s+', text):
            if f'{word} breast' in s.lower():
                return s
        return ''
    for it in items:
        for word in ['left', 'right']:
            gt_s, pr_s = _side_sentence(it['gt'], word), _side_sentence(it['pred'], word)
            b_g, b_p = extract_birads(gt_s), extract_birads(pr_s)
            d_g, d_p = extract_density(gt_s), extract_density(pr_s)
            if b_g and b_p: bg.append(int(b_g)); bp.append(int(b_p))
            if d_g and d_p: dg.append(d_g); dp.append(d_p)
    print(f'\nBI-RADS:  acc={accuracy_score(bg, bp):.3f}, weighted kappa={cohen_kappa_score(bg, bp, weights="quadratic"):.3f}  ({len(bg)} valid)')
    print(f'Density:  acc={accuracy_score(dg, dp):.3f}, weighted kappa={cohen_kappa_score(dg, dp, weights="quadratic"):.3f}  ({len(dg)} valid)')

# grounding IoU
if 'single_finding_grounding' in by_task:
    items = by_task['single_finding_grounding']
    ious = []
    for it in items:
        gt_b = extract_boxes(it['gt'])
        pr_b = extract_boxes(it['pred'])
        if not gt_b or not pr_b:
            ious.append(0.0); continue
        ious.append(iou(gt_b[0], pr_b[0]))
    ious = np.array(ious)
    print(f'\nGrounding: mean IoU = {ious.mean():.3f}, IoU@0.5 = {(ious >= 0.5).mean():.3f}  ({len(ious)} cases)')

# report extraction
if 'grounded_report' in by_task:
    items = by_task['grounded_report']
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    print(f'\ngrounded_report (finding extraction from report): macro F1 = {f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}')

## 12. Rescoring with side-aware parsing
Run on `preds_grounded_multiimage.jsonl` (first training run, all 2,000 test pairs). This output is the source of the bilateral results table in the README.

In [ ]:
# Standalone rescoring of PRED_OUT (the README's bilateral results table comes from this output).
# Adds two things the eval above doesn't: a "no findings" sentence only cancels itself (not the
# other breast's findings), and grounding checks whether the model named the correct image.
import json, re
import numpy as np
from sklearn.metrics import accuracy_score, cohen_kappa_score

CLASSES = ['Mass', 'Suspicious Calcification', 'Asymmetry', 'Architectural Distortion',
           'Suspicious Lymph Node', 'Other Suspicious Signs']
KW = {'Mass': ['mass'], 'Suspicious Calcification': ['calcification', 'calcified'],
      'Asymmetry': ['asymmetry'], 'Architectural Distortion': ['architectural distortion', 'distortion'],
      'Suspicious Lymph Node': ['lymph node'],
      'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction']}
NEG = ['no suspicious mammographic finding', 'no suspicious finding', 'no finding',
       'shows no suspicious', 'no suspicious abnormality']
BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])\b', re.I)   # \b = the density-bug fix
DENSITY_RE = re.compile(r'density\s*([abcd])\b', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>', re.I)
IMG_RE     = re.compile(r'image\s*([12])', re.I)

def sents(t): return [s for s in re.split(r'(?<=\.)\s+', str(t)) if s.strip()]

def findings(t):  # a "no findings" sentence only cancels itself, not the other breast's findings
    out = set()
    for s in sents(t):
        low = s.lower()
        if any(n in low for n in NEG): continue
        out |= {g for g, ks in KW.items() if any(k in low for k in ks)}
    return out

def asym_sides(t):
    sides = set()
    for s in sents(t):
        low = s.lower()
        if 'asymmetry' in low and not any(n in low for n in NEG):
            if 'left' in low or 'image 1' in low: sides.add('L')
            if 'right' in low or 'image 2' in low: sides.add('R')
    return sides

def per_side(t):  # {'L': (birads, density), 'R': (...)}
    out = {}
    for side, word in [('L', 'left'), ('R', 'right')]:
        s = next((x for x in sents(t) if f'{word} breast' in x.lower()), None)
        b = BIRADS_RE.search(s) if s else None
        d = DENSITY_RE.search(s) if s else None
        out[side] = (b.group(1) if b else None, d.group(1).upper() if d else None)
    return out

def first_box(t):  # first box in the answer + the image it names (1 = left, 2 = right)
    b = BOX_RE.findall(str(t)); m = IMG_RE.search(str(t))
    return (list(map(int, b[0])) if b else None), (int(m.group(1)) if m else None)

def iou(a, b):
    ix, iy = max(0, min(a[2], b[2]) - max(a[0], b[0])), max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0.0

def f1_table(items):
    yt = np.array([[g in findings(r['gt']) for g in CLASSES] for r in items])
    yp = np.array([[g in findings(r['pred']) for g in CLASSES] for r in items])
    res = {}
    for j, g in enumerate(CLASSES):
        tp, fp, fn = (yt[:, j] & yp[:, j]).sum(), (~yt[:, j] & yp[:, j]).sum(), (yt[:, j] & ~yp[:, j]).sum()
        res[g] = (2*tp / (2*tp + fp + fn) if (2*tp + fp + fn) else 0.0, int(yt[:, j].sum()))
    return res

rows = [json.loads(l) for l in open(PRED_OUT)]
by = {}
for r in rows: by.setdefault(r['task'], []).append(r)
print(f'{PRED_OUT.name}: ' + ', '.join(f'{k}={len(v)}' for k, v in by.items()))

if 'finding_classification' in by:
    res = f1_table(by['finding_classification'])
    print(f"\nFinding classification (per pair, any side): macro F1 = {np.mean([f for f, _ in res.values()]):.3f}")
    for g, (f, n) in res.items(): print(f'  {g:26s} F1={f:.3f}  (n={n})')
    pairs = [(asym_sides(r['gt']), asym_sides(r['pred'])) for r in by['finding_classification']]
    pairs = [(g, p) for g, p in pairs if g and p]
    print(f'  Asymmetry side correct when both mention it: {sum(g == p for g, p in pairs)}/{len(pairs)}')

if 'birads_density' in by:
    bg, bp, dg, dp, miss = [], [], [], [], 0
    for r in by['birads_density']:
        g, p = per_side(r['gt']), per_side(r['pred'])
        for s in 'LR':
            if g[s][0] is None: continue
            if None in p[s]: miss += 1; continue
            bg.append(g[s][0]); bp.append(p[s][0]); dg.append(g[s][1]); dp.append(p[s][1])
    print(f'\nPer breast ({len(bg)} scored, {miss} unparseable predictions):')
    print(f'  BI-RADS  acc={accuracy_score(bg, bp):.3f}  kappa_w={cohen_kappa_score(bg, bp, weights="quadratic"):.3f}')
    print(f'  Density  acc={accuracy_score(dg, dp):.3f}  kappa_w={cohen_kappa_score(dg, dp, weights="quadratic"):.3f}')

if 'single_finding_grounding' in by:
    ious, strict, ok = [], [], 0
    for r in by['single_finding_grounding']:
        (gb, gi), (pb, pi) = first_box(r['gt']), first_box(r['pred'])
        v = iou(gb, pb) if gb and pb else 0.0
        same = gi is not None and gi == pi
        ious.append(v); strict.append(v if same else 0.0); ok += same
    ious = np.array(ious)
    print(f'\nGrounding ({len(ious)}): mean IoU={ious.mean():.3f}  IoU@0.5={(ious >= 0.5).mean():.3f}  '
          f'correct image={ok/len(ious):.3f}  mean IoU if correct image required={np.mean(strict):.3f}')

if 'grounded_report' in by:
    res = f1_table(by['grounded_report'])
    print(f"\nGrounded report (findings read back out): macro F1 = {np.mean([f for f, _ in res.values()]):.3f}")

preds_grounded_multiimage.jsonl: finding_classification=2000, birads_density=2000, grounded_report=2000, single_finding_grounding=482

Finding classification (per pair, any side): macro F1 = 0.161
  Mass                       F1=0.415  (n=214)
  Suspicious Calcification   F1=0.503  (n=105)
  Asymmetry                  F1=0.047  (n=78)
  Architectural Distortion   F1=0.000  (n=24)
  Suspicious Lymph Node      F1=0.000  (n=10)
  Other Suspicious Signs     F1=0.000  (n=14)
  Asymmetry side correct when both mention it: 2/2

Per breast (4000 scored, 0 unparseable predictions):
  BI-RADS  acc=0.712  kappa_w=0.471
  Density  acc=0.830  kappa_w=0.578

Grounding (482): mean IoU=0.228  IoU@0.5=0.212  correct image=0.801  mean IoU if correct image required=0.227

Grounded report (findings read back out): macro F1 = 0.186
